# DBSCAN — A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise (1996)

_Papers · Classical ML_

**Grow clusters by chaining together dense neighborhoods — finding clusters of any shape and flagging the leftover points as noise.**

---

This notebook is a practice scaffold for the **DBSCAN — A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise (1996)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. This lesson trains on **synthetic** data built by `make_moons(...)` in the code below. There are no real-world column names — the features are unnamed numeric dimensions. As the cell runs, watch the shape of `X` (rows × features) and the labels in `y`.

## Reference implementation — NumPy + scikit-learn

DBSCAN grows clusters by chaining together dense neighborhoods. We build it from scratch in four steps: (1) the neighborhood query and a worked core-point check, (2) the main DBSCAN loop that grows clusters, (3) verifying our version matches scikit-learn on noisy two-moons data, and (4) contrasting it with k-means, which cannot follow non-convex shapes.

### Step 1 — The Eps-neighborhood query and the core-point test

DBSCAN's primitive is the **Eps-neighborhood** (Def 1): all points within distance `eps` of a given point, including the point itself. A point is a **core point** (Def 2) if its neighborhood holds at least `min_pts` points. We test this on five hand-placed points: A sits among a dense cluster (core), while E is off alone (not core, so noise).

In [ ]:
import numpy as np
from collections import deque

def region_query(X, i, eps):
    # Def 1: Eps-neighborhood of point i (includes i itself).
    d = np.linalg.norm(X - X[i], axis=1)
    return np.where(d <= eps)[0]

# Worked core-point example (Eps=1.0, MinPts=4).
P = np.array([[0,0],[0.5,0],[0,0.6],[0.8,0.6],[5,5]], dtype=float)  # A,B,C,D,E

nbrs_A = region_query(P, 0, 1.0)
print("|N_Eps(A)| =", len(nbrs_A), "-> A core?", len(nbrs_A) >= 4)   # 4 -> True

nbrs_E = region_query(P, 4, 1.0)
print("|N_Eps(E)| =", len(nbrs_E), "-> E core?", len(nbrs_E) >= 4)   # 1 -> False (noise)

### Step 2 — The DBSCAN loop: start and grow clusters

We sweep over every point. If a point isn't a core point, it's tentatively labeled noise (it may later be reclaimed as a **border** point). When we hit a core point we start a new cluster and **expand** it: a queue of seed neighbors grows the cluster outward, pulling in every density-reachable point and continuing the chain whenever a seed is itself a core point.

In [ ]:
def my_dbscan(X, eps, min_pts):
    n = len(X)
    UNVISITED, NOISE = 0, -1
    labels = np.full(n, UNVISITED)   # 0 = unvisited; -1 = noise; >=1 = cluster id
    cid = 0
    for i in range(n):
        if labels[i] != UNVISITED:
            continue
        nbrs = region_query(X, i, eps)
        if len(nbrs) < min_pts:      # Def 2 fails: not a core point
            labels[i] = NOISE        # tentative noise (may be reclaimed as border)
            continue
        cid += 1                     # start a new cluster
        labels[i] = cid
        seeds = deque(nbrs.tolist()) # ExpandCluster: grow along density-reachability
        while seeds:
            j = seeds.popleft()
            if labels[j] == NOISE:
                labels[j] = cid      # border point reclaimed
            if labels[j] != UNVISITED:
                continue
            labels[j] = cid
            j_nbrs = region_query(X, j, eps)
            if len(j_nbrs) >= min_pts:           # j is core -> keep expanding
                seeds.extend(j_nbrs.tolist())
    return labels

### Step 3 — Match scikit-learn on noisy two-moons

To check correctness we build two interleaving crescents with `make_moons`, sprinkle in 20 scattered outliers, and run both our DBSCAN and scikit-learn's. Cluster ids are arbitrary (and border-point assignment is order-dependent), so we compare them as a **partition** with the Adjusted Rand Index — `ARI = 1.0` means identical clustering up to relabeling.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

X, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
rng = np.random.default_rng(0)
noise_pts = rng.uniform(X.min(0)-0.3, X.max(0)+0.3, size=(20, 2))  # scattered outliers
X = np.vstack([X, noise_pts])

eps, min_pts = 0.2, 5
mine = my_dbscan(X, eps, min_pts)
ref  = DBSCAN(eps=eps, min_samples=min_pts).fit_predict(X)

# Adjusted Rand Index = 1.0 means identical clustering up to relabeling.
ari = adjusted_rand_score(mine, ref)
print("ARI(mine, sklearn):", round(ari, 6))          # expect 1.0
print("same #clusters:", len(set(mine)) == len(set(ref)),
      "| mine noise:", int((mine == -1).sum()),
      "sklearn noise:", int((ref == -1).sum()))

### Step 4 — Contrast with k-means

k-means assigns each point to its nearest center, so every cluster is a convex Voronoi cell — it cannot bend around a crescent. Running it with `k=2` on the same data, it slices the moons into two convex halves, whereas DBSCAN recovers the two true crescents plus the scattered noise.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
# DBSCAN recovers 2 crescents + noise; k-means cuts them into 2 convex halves.
dbscan_clusters = len(set(mine)) - (1 if -1 in mine else 0)
print("DBSCAN clusters (excl. noise):", dbscan_clusters)
print("k-means clusters:", len(set(km)), "(convex Voronoi cells -> splits the moons)")

## Visualize the data & results

_On two interleaving moons with scattered noise, does our from-scratch DBSCAN recover the two non-convex crescents and flag the outliers, where k-means (forced into convex Voronoi cells) cannot? And what do too-small / too-large Eps do?_

### Step 1 — Rebuild DBSCAN and confirm it matches scikit-learn

For a self-contained visualization we re-import everything and restate a compact `my_dbscan`. We regenerate the same noisy two-moons and confirm our clustering matches scikit-learn (ARI = 1.0), recovering two clusters plus noise.

In [ ]:
import numpy as np
from collections import deque
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score

def region_query(X, i, eps):
    return np.where(np.linalg.norm(X - X[i], axis=1) <= eps)[0]

def my_dbscan(X, eps, min_pts):
    n = len(X)
    labels = np.zeros(n, int)
    cid = 0
    for i in range(n):
        if labels[i] != 0:
            continue
        nbrs = region_query(X, i, eps)
        if len(nbrs) < min_pts:
            labels[i] = -1
            continue
        cid += 1
        labels[i] = cid
        seeds = deque(nbrs.tolist())
        while seeds:
            j = seeds.popleft()
            if labels[j] == -1:
                labels[j] = cid
            if labels[j] != 0:
                continue
            labels[j] = cid
            jn = region_query(X, j, eps)
            if len(jn) >= min_pts:
                seeds.extend(jn.tolist())
    return labels

X, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
rng = np.random.default_rng(0)
X = np.vstack([X, rng.uniform(X.min(0)-0.3, X.max(0)+0.3, size=(20,2))])

mine = my_dbscan(X, 0.2, 5)
ref  = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
print("ARI vs sklearn:", round(adjusted_rand_score(mine, ref), 6))   # 1.0
print("DBSCAN clusters:", len(set(mine)) - (1 if -1 in mine else 0), # 2
      "| noise:", int((mine == -1).sum()))

### Step 2 — Compare to k-means and ablate Eps

Now the failure modes. k-means again splits the moons into two convex halves. Then we sweep `eps`: too small (0.01) and almost nothing reaches `min_pts` neighbors, so every point becomes noise; too large (2.0) and the neighborhoods bridge the gap, merging both moons into a single cluster. The right `eps` (~0.2, from the sorted k-dist 'valley') sits between these extremes.

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
print("k-means clusters:", len(set(km)), "(splits the moons)")        # 2 convex halves

# Ablation: eps too small / too large.
small = my_dbscan(X, 0.01, 5)
big   = my_dbscan(X, 2.0,  5)

small_clusters = len(set(small)) - (1 if -1 in small else 0)
print("eps=0.01 -> clusters:", small_clusters,
      "noise:", int((small == -1).sum()))    # 0 clusters, all noise

big_clusters = len(set(big)) - (1 if -1 in big else 0)
print("eps=2.0  -> clusters:", big_clusters)  # 1 (merged)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With $Eps=1.5$, $MinPts=3$, is the point $P=(0,0)$ a core point given neighbors at $(1,0)$, $(0,1)$, and $(2,2)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute distances from $P$: $dist=1.0$, $1.0$, and $\sqrt{8}\approx 2.83$. — _Definition 1 keeps only points within $Eps=1.5$._
- Keep those $\le 1.5$: $(1,0)$ and $(0,1)$. Add $P$ itself. — _A point is in its own Eps-neighborhood._
- Count: $|N_{Eps}(P)|=3$ (namely $P$, $(1,0)$, $(0,1)$). — _$(2,2)$ is too far, so it is excluded._

**Answer:** $|N_{Eps}(P)|=3 \ge MinPts=3$, so $P$ is a core point. The point at $(2,2)$ is not in $P$'s neighborhood; whether it is noise depends on whether any core point reaches it.

</details>

**Problem 2.** Ablation — $Eps$ too small. In the CODE, set $Eps$ far below the typical nearest-neighbor distance (e.g. $0.01$) and re-run on the two moons. What labels do you expect?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Lower $Eps$ to $0.01$ and keep $MinPts=5$. — _Shrinks every Eps-neighborhood toward just the point itself._
- Check how many points pass the core test $|N_{Eps}(p)|\ge MinPts$. — _With a tiny radius almost none gather $5$ neighbors._
- Read off the labels. — _No core points means no cluster can start or grow._

**Answer:** Almost no point reaches $MinPts$ neighbors, so there are essentially no core points and DBSCAN labels nearly everything noise ($-1$): zero clusters found. In our small run (CODEVIZ ablation) $Eps=0.01$ yields 0 clusters and all points noise — the classic 'Eps too small' failure.

</details>

**Problem 3.** Ablation — $Eps$ too large. Now set $Eps$ much larger than the gap between the two moons (e.g. $2.0$). What happens to the two crescents?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Raise $Eps$ to $2.0$, keep $MinPts=5$. — _Enlarges every neighborhood so distant points become 'close'._
- Trace density-reachability between the two moons. — _If a chain of core points bridges the gap, the moons connect._
- Count the resulting clusters. — _Over-merging collapses distinct groups into one._

**Answer:** With $Eps=2.0$ the neighborhoods are large enough that core points bridge the gap between the moons (and swallow the noise), so DBSCAN returns a single cluster instead of two — the 'Eps too large' failure. The right $Eps$ (from the sorted $k$-dist 'valley', Section 4.2) sits between these extremes; in our run $Eps\approx 0.2$ recovers exactly the two moons plus noise.

</details>

**Problem 4.** Why does k-means fail on the two moons where DBSCAN succeeds?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall k-means assigns each point to its nearest center, so each cluster is a Voronoi cell. — _Voronoi cells are convex — no dents._
- Note the two moons are interleaving crescents — non-convex, and partly wrapped around each other. — _A convex region cannot follow a crescent._
- DBSCAN instead grows clusters along chains of dense neighborhoods (density-reachability). — _Density-reachability follows any shape, one dense step at a time._

**Answer:** k-means must cut the plane into convex Voronoi cells, so it slices each crescent in half and mixes the two moons. DBSCAN follows the local density, snaking along each crescent via density-reachable core points, so it recovers the true non-convex clusters and leaves the scattered points as noise — exactly the qualitative result the paper reports (DBSCAN finds arbitrary-shape clusters that partitioning methods split).

</details>